# 🎯 객체 탐지 (Object Detection)

MediaPipe Object Detector를 사용하여 이미지 내의 **객체를 탐지**합니다.

```bash
pip install mediapipe opencv-python numpy
```

| 항목 | 내용 |
|------|------|
| **태스크** | `ObjectDetector` |
| **모델** | `models/object_detector.tflite` (EfficientDet-Lite0) |
| **탐지 클래스** | 80개 (COCO 데이터셋: 사람, 자동차, 동물 등) |
| **출력** | 바운딩 박스, 클래스명, 신뢰도 |

---
## 1. COCO 80개 클래스

EfficientDet-Lite0 모델은 COCO 데이터셋의 80개 객체 클래스를 인식합니다.

주요 클래스 (일부):

| ID | 클래스 | ID | 클래스 | ID | 클래스 |
|----|--------|----|--------|----|--------|
| 0 | person | 2 | car | 16 | dog |
| 1 | bicycle | 3 | motorcycle | 17 | horse |
| 5 | bus | 7 | truck | 25 | umbrella |
| 8 | traffic light | 15 | cat | 62 | couch |
| 9 | fire hydrant | | | 73 | laptop |

전체 80개 클래스는 [COCO 라벨](https://cocodataset.org/#explore?id=80)에서 확인할 수 있습니다.

In [ ]:
# 객체 탐지 — 정적 이미지 처리
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np

# 모델 로드
model_path = 'models/object_detector.tflite'
base_options = mp.tasks.BaseOptions(model_asset_path=model_path)
options = vision.ObjectDetectorOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    max_results=5,              # 최대 탐지 객체 수
    score_threshold=0.3,        # 최소 신뢰도 (30%)
    category_allowlist=None,    # 특정 클래스만 탐지하려면 ['person'] 등 지정
)

# 테스트 이미지 생성
test_img = np.zeros((480, 640, 3), dtype=np.uint8)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=test_img)

# 추론 실행
with vision.ObjectDetector.create_from_options(options) as detector:
    result = detector.detect(mp_image)
    
    print(f'탐지된 객체 수: {len(result.detections)}')
    
    if result.detections:
        for i, detection in enumerate(result.detections):
            bbox = detection.bounding_box
            category = detection.categories[0]
            print(f'  [{i}] {category.category_name} '
                  f'(신뢰도: {category.score:.2f}) '
                  f'박스: ({bbox.origin_x}, {bbox.origin_y}) '
                  f'{bbox.width}x{bbox.height}')
    else:
        print('  탐지된 객체 없음 (빈 이미지이므로 정상)')

# 탐지 결과 구조 확인
print(f'\n결과 타입: {type(result).__name__}')
print(f'속성: {[a for a in dir(result) if not a.startswith("_")]}')

---
## 2. 바운딩 박스 시각화

탐지된 객체에 바운딩 박스와 라벨을 그립니다.
클래스별로 다른 색상을 사용하여 가독성을 높입니다.

In [ ]:
# 객체 탐지 시각화 함수
import cv2
import numpy as np

# 클래스별 색상 (일부)
CLASS_COLORS = {
    'person': (0, 255, 0),     # 초록
    'car': (0, 0, 255),        # 빨강
    'cat': (255, 0, 0),        # 파랑
    'dog': (255, 165, 0),      # 주황
    'bicycle': (255, 255, 0),  # 시안
    'motorcycle': (0, 255, 255),  # 노랑
    'bus': (255, 0, 255),      # 마젠타
    'truck': (128, 0, 128),    # 보라
}
DEFAULT_COLOR = (200, 200, 200)

def draw_detections(image, detections):
    """탐지 결과를 이미지 위에 그립니다."""
    annotated = image.copy()
    
    for detection in detections:
        bbox = detection.bounding_box
        category = detection.categories[0]
        
        # 색상 선택
        color = CLASS_COLORS.get(category.category_name, DEFAULT_COLOR)
        
        # 바운딩 박스
        x1, y1 = bbox.origin_x, bbox.origin_y
        x2, y2 = x1 + bbox.width, y1 + bbox.height
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        
        # 라벨 텍스트
        label = f'{category.category_name} {category.score:.2f}'
        
        # 라벨 배경
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
        cv2.rectangle(annotated, (x1, y1 - th - 5), (x1 + tw, y1), color, -1)
        cv2.putText(annotated, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    return annotated

# 가상 탐지 결과로 시각화 테스트
class FakeBBox:
    def __init__(self, x, y, w, h):
        self.origin_x, self.origin_y = x, y
        self.width, self.height = w, h

class FakeCategory:
    def __init__(self, name, score):
        self.category_name = name
        self.score = score

class FakeDetection:
    def __init__(self, bbox, cat_name, score):
        self.bounding_box = bbox
        self.categories = [FakeCategory(cat_name, score)]

# 가상 감지 결과 생성
fake_detections = [
    FakeDetection(FakeBBox(100, 150, 200, 300), 'person', 0.92),
    FakeDetection(FakeBBox(350, 200, 150, 120), 'car', 0.85),
    FakeDetection(FakeBBox(520, 100, 80, 80), 'dog', 0.70),
]

test_img = np.zeros((480, 640, 3), dtype=np.uint8)
result_img = draw_detections(test_img, fake_detections)
print(f'시각화 완료: {result_img.shape}')
print('웹캠 환경에서 실제 이미지로 확인하세요.')

---
## 3. 카테고리 필터링

`category_allowlist`를 사용하여 특정 클래스만 탐지할 수 있습니다.
`category_denylist`로 특정 클래스를 제외할 수도 있습니다.

In [ ]:
# 카테고리 필터링 예제
import mediapipe as mp
from mediapipe.tasks.python import vision

# 사람만 탐지
base_options = mp.tasks.BaseOptions(model_asset_path='models/object_detector.tflite')
options_person = vision.ObjectDetectorOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    max_results=10,
    score_threshold=0.3,
    category_allowlist=['person']  # 사람만
)
print('사람만 탐지하는 감지기 설정 완료')

# 차량 관련만 탐지
options_vehicle = vision.ObjectDetectorOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    max_results=10,
    score_threshold=0.3,
    category_allowlist=['car', 'bus', 'truck', 'motorcycle', 'bicycle']
)
print('차량만 탐지하는 감지기 설정 완료')

# 사람 제외하고 탐지
options_no_person = vision.ObjectDetectorOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    max_results=10,
    score_threshold=0.3,
    category_denylist=['person']  # 사람 제외
)
print('사람을 제외하고 탐지하는 감지기 설정 완료')

---
## 4. 웹캠 실시간 객체 탐지

웹캠에서 실시간으로 객체를 탐지하고 시각화합니다.

In [ ]:
# 웹캠 실시간 객체 탐지 (참고용 — 웹캠 환경에서 실행)
# import mediapipe as mp
# from mediapipe.tasks.python import vision
# import cv2
# import time

# model_path = 'models/object_detector.tflite'
# base_options = mp.tasks.BaseOptions(model_asset_path=model_path)
# options = vision.ObjectDetectorOptions(
#     base_options=base_options,
#     running_mode=vision.RunningMode.LIVE_STREAM,
#     max_results=5,
#     score_threshold=0.4,
#     result_callback=lambda r, img, ts: None
# )

# cap = cv2.VideoCapture(0)
# timestamp = 0

# with vision.ObjectDetector.create_from_options(options) as detector:
#     while cap.isOpened():
#         ret, frame = cap.read()
#         if not ret: break
#         
#         mp_image = mp.Image(mp.ImageFormat.SRGB, frame)
#         timestamp = int(time.time() * 1000)
#         detector.detect_async(mp_image, timestamp)
#         
#         if detector.latest_results and detector.latest_results.detections:
#             frame = draw_detections(frame, detector.latest_results.detections)
#             count = len(detector.latest_results.detections)
#             cv2.putText(frame, f'Objects: {count}', (10, 30),
#                         cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
#         
#         cv2.imshow('Object Detection', frame)
#         if cv2.waitKey(1) & 0xFF == ord('q'): break
# cap.release()
# cv2.destroyAllWindows()
print('웹캠 코드는 주석 처리됨 — 웹캠 환경에서 주석 해제 후 실행')

---
## 5. 실전: 객체 카운터

탐지된 객체의 클래스별 개수를 세어 통계를 출력합니다.
보안 카메라, 교통 모니터링 등에 활용할 수 있습니다.

In [ ]:
# 객체 카운터 함수
from collections import Counter

def count_objects(detections):
    """탐지된 객체를 클래스별로 카운트합니다."""
    counts = Counter()
    for detection in detections:
        if detection.categories:
            cat_name = detection.categories[0].category_name
            counts[cat_name] += 1
    return counts

def format_counts(counts):
    """카운트 결과를 문자열로 포맷합니다."""
    if not counts:
        return 'No objects'
    parts = [f'{name}: {count}' for name, count in counts.most_common()]
    return ' | '.join(parts)

# 가상 탐지 결과로 테스트
fake_multi = [
    FakeDetection(FakeBBox(100, 150, 200, 300), 'person', 0.92),
    FakeDetection(FakeBBox(350, 200, 100, 200), 'person', 0.85),
    FakeDetection(FakeBBox(500, 180, 120, 100), 'car', 0.78),
    FakeDetection(FakeBBox(50, 300, 80, 80), 'dog', 0.65),
    FakeDetection(FakeBBox(250, 350, 60, 60), 'person', 0.55),
    FakeDetection(FakeBBox(400, 320, 90, 90), 'car', 0.45),
]

counts = count_objects(fake_multi)
print('객체 카운트:')
for name, count in counts.most_common():
    print(f'  {name}: {count}개')
print(f'\n포맷 결과: {format_counts(counts)}')
print(f'총 객체 수: {sum(counts.values())}개')

---
## 🎯 정리

| 기능 | 코드 | 설명 |
|------|------|------|
| 탐지 실행 | `detector.detect(mp_image)` | 이미지에서 객체 탐지 |
| 결과 | `result.detections` | 탐지된 객체 리스트 |
| 바운딩박스 | `detection.bounding_box` | origin_x, origin_y, width, height |
| 클래스 | `detection.categories[0]` | category_name, score |
| 최대 수 | `max_results=5` | 최대 탐지 객체 수 |
| 신뢰도 | `score_threshold=0.3` | 30% 이상만 탐지 |
| 허용 목록 | `category_allowlist=['person']` | 특정 클래스만 |
| 제외 목록 | `category_denylist=['person']` | 특정 클래스 제외 |

---
## 🎯 연습 문제

1. **사람 감지 경고**: 실시간으로 사람이 감지되면 화면에 'PERSON DETECTED!' 경고를 표시하는 코드를 작성하세요.

2. **영역별 카운트**: 이미지를 좌/우로 나누고 각 영역에 있는 객체 수를 세어 표시하세요. (힌트: 바운딩 박스의 중심 x좌표로 영역 판단)

3. **FPS 측정**: 실시간 객체 탐지의 FPS를 측정하고 화면에 표시하는 코드를 작성하세요. (힌트: `time.time()`로 프레임 간 시간 측정)

4. **객체 추적**: 같은 클래스의 객체가 연속된 프레임에서 가장 가까운 위치에 있으면 같은 객체로 추적하는 간단한 추적기를 구현하세요. (힌트: 이전 프레임의 바운딩 박스 중심과 현재 프레임의 중심 거리 비교)